## This notebook is meant to be a module to read and filter and prepare the enformer and AF tables of mohan
- last column: AF=...;AC=... (remove AF= and split to two column)
- add a header
- make it a good pandas df to merge with the header
- see "/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/MPRAlm/scripts/reading_enformer_tsv.py" as well

In [97]:
import yaml
import os
import pandas as pd


# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/config/config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [108]:
## functions for this type of files

def load_df_add_columns(file_path):
    """Add column names and return dataframe"""
    enformer_af_df = pd.read_csv(file_path, sep="\t", header=None)
    enformer_af_df.columns = ["CHROM", "POS", "ID", "REF", "ALT", "enformer_value", "max_enformer_column",  "QUAL", "FILTER", "INFO"]
    return enformer_af_df

def seperate_columns_by_pattern(df, column_with_pattern, pattern, new_column_names, drop_original_column=True):
    """Seperates columns by pattern and adds new column names"""
    extracted_df = df[column_with_pattern].str.extract(pattern)
    extracted_df.columns = new_column_names
    if drop_original_column:
        df = df.drop(column_with_pattern, axis=1)
    df = pd.concat([df, extracted_df], axis=1)
    return df

def get_AF_AC_from_INFO_column(enformer_df):
    """Adds AF and AC columns instead of INFO column"""
    df = seperate_columns_by_pattern(enformer_df, "INFO", r'AF=(?P<AF>[\d.e-]+);AC=(?P<AC>\d+)', ["AF", "AC"], drop_original_column=True)
    return df

def get_rows(df):
    """Get a specific number of rows"""
    pass

def get_higher_lower_subset(enformer_df, variant_sum=5000, higher_fraction = 0.7, lower_fraction = 0.15, column="enformer_value"):
    """Get higher 70% and lower 15% of the enformer df sorted by column such that it sums up to 5000"""
    # sort the df 
    enformer_sorted = enformer_df.sort_values(by=column, ascending=False)
    
    # get top 0.7 * 5000 rows
    top_rows = int(variant_sum * higher_fraction)
    bottom_rows = int(variant_sum * lower_fraction)
    enformer_high = enformer_sorted.iloc[:top_rows]
    
    enformer_low = enformer_sorted.iloc[len(enformer_sorted)-bottom_rows:]
    #! TODO: debugging enformer high: only 2500 rows in current table (at least 5000 are expected)
    print("expected number of enformer high: ", top_rows)
    print("current number of enformer high: ", len(enformer_high))
    print("expected number of enformer low: ", bottom_rows)
    print("current number of enformer low: ", len(enformer_low))
    # check if len(enformer_low) equals bottom_rows
    if len(enformer_high) != top_rows:
        print("Top rows does not work")
    else:
        print("Top rows work")
    if len(enformer_low) == bottom_rows:
        print("bottom rows work")
    else: 
        print("bottom rows do not work")
    return enformer_high, enformer_low
    

In [112]:
_verbose = config["verbose"]
enformer_prediction_value = "enformer_value"
variant_sum=5000 
higher_fraction = 0.7
lower_fraction = 0.15
# read as tab separated file 
for name in config["condition_names"]:
    print(f"----{name}------")
    file_name = name + "." + ".".join(config["default_name"].split(".")[1:]) # pathing the file name from the config
    enformer_af_path = os.path.join(config["enformer_results_dir"], file_name)
    print(enformer_af_path)
    # load table and add columns
    enformer_af_df = load_df_add_columns(enformer_af_path)  
    # print(enformer_af_df.head())
    ## change info column AF=...;AC=... into two columns (AF, AC)
    enformer_af_ac_df = get_AF_AC_from_INFO_column(enformer_af_df)
    # sort by ac (to test)
    enformer_sorted = enformer_af_ac_df.sort_values(by=enformer_prediction_value, ascending=False)
    
    # take num * fraction first / last elements
    enformer_high, enformer_low = get_higher_lower_subset(enformer_df=enformer_af_ac_df, column=enformer_prediction_value)
    #! TODO: enformer_high set is broken (intitial table has a length of 2500 (expected 5000)) check if you downloaded the correct one
    # write enformer_low
    low_number = ""
    high_number = ""
    if _verbose:
        # expected number _ real number
        low_number = "_" + str(int(variant_sum * lower_fraction)) + "_" + str(len(enformer_low))
        high_number = "_" + str(int(variant_sum * higher_fraction)) + "_" + str(len(enformer_high))
    output_low = ".".join(file_name.split(".")[:-1]) + f"{low_number}_low.tsv"
    output_high = ".".join(file_name.split(".")[:-1]) + f"{high_number}_high.tsv"
    enformer_low.to_csv(os.path.join(config["output_dir"], output_low), sep="\t", header=True, index=False)
    enformer_high.to_csv(os.path.join(config["output_dir"], output_high), sep="\t", header=True, index=False)


----cardiac------
/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/enformer_results/cardiac.ultra-rare_variants.vcf
expected number of enformer high:  3500
current number of enformer high:  2500
expected number of enformer low:  750
current number of enformer low:  750
Top rows does not work
bottom rows work
----cava------
/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/enformer_results/cava.ultra-rare_variants.vcf
expected number of enformer high:  3500
current number of enformer high:  2501
expected number of enformer low:  750
current number of enformer low:  750
Top rows does not work
bottom rows work
----neuro------
/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/enformer_results/neuro.ultra-rare_variants.vcf
expected number of enformer high:  3500
current number of enformer high:  2501
expected number of enformer low:  750
current number of enformer low:  750
Top rows does not work
bottom rows work
----random------
/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/enformer_r

## Match the enformer low (working) with the significant results of MPRAlm

In [115]:
mpra_results = "/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/MPRAlm/results/NGN2_output/mpralm/toptable_2701_all_NGN2.feather"
mpra_df = pd.read_feather(mpra_results)

enformer_low_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/04_MPRAlm/enformer_results/high_low_tables/cardiac.ultra-rare_variants_750_750_low.tsv"
enformer_low = pd.read_csv(enformer_low_path, sep="\t")
print(enformer_low.head())

   CHROM        POS                                                 ID REF  \
0   chr5  180608335  FLT4|ENSG00000037280.16|EH38E3684204|5-1806083...   C   
1  chr16   54080804  FTO|ENSG00000140718.21|EH38E1816677|16-5408080...   G   
2   chr1   15680797  PLEKHM2|ENSG00000116786.13|EH38E1321744|1-1568...   G   
3   chr2  105447631  FHL2|ENSG00000115641.19|EH38E2021840|2-1054476...   T   
4   chr1  201421798  TNNT2|ENSG00000118194.21|EH38E2857804|1-201421...   A   

  ALT  enformer_value       max_enformer_column  QUAL FILTER        AF   AC  
0   A       384.77628           425_DNASE:Caki2     1   PASS  0.000013    2  
1   A       384.43860           234_DNASE:HepG2     1   PASS  0.000039    6  
2   A       384.17420            625_DNASE:K562     1   PASS  0.000013    2  
3   G       383.85290            636_DNASE:PC-9     1   PASS  0.000020    3  
4   T       383.79480  282_DNASE:gastrocnemius      1   PASS  0.003134  473  


In [118]:
p_sig_toptable = mpra_df[mpra_df["adj.P.Val"] < 0.05]
p_sig_toptable.head()
#! TODO: table from chrom pos ref alt to header and id for each sequence
p_sig_toptable.merge(enformer)

,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id
0,1.405459,1.779925,16.343686,5.966671e-60,1.259087e-55,123.851984,cardiac_neuro_cava_random:TRIO|ENSG00000038382...
1,1.461423,0.673326,14.537740,7.982489e-48,8.422324e-44,95.885456,cardiac_neuro_cava_random:TRIO|ENSG00000038382...
2,1.528669,0.951499,11.926336,9.168757e-33,6.449304e-29,61.312948,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...
3,1.026733,1.106780,10.681717,1.289127e-26,6.800792e-23,48.979467,cardiac_neuro_cava_random:SLC1A2|ENSG000001104...
4,-1.197234,0.772053,-10.386340,2.963358e-25,1.250656e-21,45.452959,cardiac_neuro_cava_random:DISC1|ENSG0000016294...
